# Error Case 9 — 500 Internal Server Error: NullPointerException

**Error:** `500 Internal Server Error`  
**Root cause:** Missing `endpointInternal` in catalog `storageConfigInfo`  
**Fix:** Add `endpointInternal` (K8s internal DNS for MinIO)

## Flow
```
GET /catalog/v1/{catalog}/namespaces (or any catalog op)
  → Polaris resolves storage config
  → tries endpointInternal for internal MinIO call
  → endpointInternal is null
  → NullPointerException → 500
```

In [13]:
import sys
sys.path.insert(0, '..')
from polaris_test_utils import *

cleanup()
print('✅ Setup complete')

🧹 Cleaning up test resources...
✅ Cleanup done
✅ Setup complete


In [14]:
# Cell 2: Create catalog WITHOUT endpointInternal
r1 = create_catalog_no_endpoint()
print_response(r1, 'Create catalog without endpointInternal')
assert r1.status_code == 201, r1.text
print('\n⚠️  Catalog created — but will fail on namespace operations')


Create catalog without endpointInternal:
  Status:     201
  Request-Id: 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000372
  Body: {
  "type": "INTERNAL",
  "name": "test-error-catalog",
  "properties": {
    "default-base-location": "s3a://data-catalog-bucket/test-error-catalog/"
  },
  "createTimestamp": 1780362957323,
  "lastUpdateTimestamp": 0,
  "entityVersion": 1,
  "storageConfigInfo": {
    "endpoint": "http://192.168.139.2:9000",
    "pathStyleAccess": true,
    "storageType": "S3",
    "allowedLocations": [
      "s3a://data-catalog-bucket/test-error-catalog/",
      "s3a://data-catalog-bucket/"
    ]
  }
}

⚠️  Catalog created — but will fail on namespace operations


In [15]:
# Cell 3: Trigger 500 — table operation triggers storage resolution
# namespace create doesn't trigger it, table create does

# First create namespace (will succeed)
r_ns = requests.post(
    f'{BASE_CAT}/{TEST_CATALOG}/namespaces',
    headers=h(),
    json={'namespace': [TEST_NAMESPACE], 'properties': {}}
)
print(f'Namespace create: {r_ns.status_code}')

# Table create triggers actual storage resolution → 500
r = requests.post(
    f'{BASE_CAT}/{TEST_CATALOG}/namespaces/{TEST_NAMESPACE}/tables',
    headers=h(),
    json={
        'name': TEST_TABLE,
        'schema': {
            'type': 'struct',
            'fields': [
                {'id': 1, 'name': 'id', 'required': True,
                 'type': 'long'},
                {'id': 2, 'name': 'data', 'required': False,
                 'type': 'string'}
            ]
        },
        'write-order': {'type': 'unsorted'},
        'stage-create': False
    }
)
print_response(r, 'Table create — expected 500')
request_id = get_request_id(r)
print(f'\nPolaris-Request-Id: {request_id}')

if r.status_code == 500:
    print('\n✅ 500 confirmed — NullPointerException on table create')
elif r.status_code == 200:
    print('\n⚠️  200 — endpointInternal not required in this Polaris version')
    print('    Behavior changed from earlier versions')
    print('    NullPointerException no longer triggered by missing endpointInternal')

Namespace create: 200

Table create — expected 500:
  Status:     400
  Request-Id: 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000376
  Body: {
  "error": {
    "message": "HTTP 400 Bad Request",
    "type": "WebApplicationException",
    "code": 400
  }
}

Polaris-Request-Id: 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000376


In [16]:
# Cell 4: Find log in OpenSearch
wait_for_logs(5)
hits = search_logs(level='ERROR', message='NullPointer', minutes_ago=2)
if not hits:
    hits = search_logs(level='ERROR', message='getRawLeafEntity', minutes_ago=2)
if not hits:
    hits = search_logs(level='ERROR', minutes_ago=2, size=5)
print_logs(hits)

⏳ Waiting 5s for logs to reach OpenSearch...
⚠️  No logs found

Running debug to check OpenSearch state...
=== OpenSearch Debug ===

Indices:
  k8s-logs-2026.06.02: 17212 docs
  k8s-logs-2026.06.01: 808016 docs

Container names in index:
  benchmarks-polaris: 11950 docs
  pgpool: 6068 docs
  spark-master: 5866 docs
  coredns: 1608 docs
  fluent-bit: 774 docs
  postgresql: 206 docs

Polaris logs last 5m: 1998 docs
Latest log sample:
  @timestamp: 2026-06-02T01:16:02.369Z
  level:      DEBUG
  message:    query: SELECT id, catalog_id, parent_id, type_code, name, entity_version, sub_type_code, create_time
  mdc:        {'requestId': '57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000372', 'realmId': 'POLARIS'}
  container:  benchmarks-polaris


In [17]:
# Cell 5: Fix — delete and recreate with endpointInternal
delete_catalog()

r_fix = create_catalog()  # has endpointInternal
print_response(r_fix, 'Recreate catalog with endpointInternal')
assert r_fix.status_code == 201

# Verify — namespace create should work now
r_verify = requests.post(
    f'{BASE_CAT}/{TEST_CATALOG}/namespaces',
    headers=h(),
    json={'namespace': [TEST_NAMESPACE], 'properties': {}}
)
print(f'\nVerify namespace create: {r_verify.status_code}')
assert r_verify.status_code in [200, 201]
print('\n✅ Fixed — endpointInternal resolves NullPointerException')

cleanup()
print('\n=== Summary ===')
print('Error:    500 Internal Server Error')
print('Cause:    Missing endpointInternal in storageConfigInfo')
print('Detect:   ERROR message contains NullPointer/getRawLeafEntity')
print('Fix:      Add endpointInternal: http://benchmarks-minio.datahub-hynix.svc.cluster.local:9000')
print('Why:      Polaris uses endpointInternal for K8s-internal MinIO calls')


Recreate catalog with endpointInternal:
  Status:     409
  Request-Id: 57c3b5f9-5ff6-4e3e-aeff-d8097fe01882_0000000000000000380
  Body: {
  "error": {
    "message": "Cannot create Catalog test-error-catalog. Catalog already exists or resolution failed",
    "type": "AlreadyExistsException",
    "code": 409
  }
}


AssertionError: 